# 📊 Análise de Satisfação e Reputação na Mídia

Bem-vindo a este relatório de inteligência de dados. Nosso objetivo aqui não é apenas contar quantas vezes a marca apareceu na mídia, mas **medir a qualidade e o impacto dessas aparições**.

Para não tratarmos o Jornal Nacional e um blog de bairro como se tivessem o mesmo impacto, utilizamos uma metodologia de **Inteligência Espacial e Estatística**:

* **O Filtro Orgânico:** Removemos toda a "Publicidade" paga logo no início. Só analisamos o que a mídia falou de forma espontânea.
* **O Peso da Voz (Tier):** Veículos foram classificados como "Muito Relevantes", "Relevantes" e "Menos Relevantes". Um veículo maior tem um peso multiplicador na nossa nota. Os pesos usados são **8 / 4 / 1** — uma escala exponencial que amplifica propositalmente a diferença entre Tiers para capturar a realidade de audiência (um veículo "Muito Relevante" não é 3× maior que um "Menos Relevante" — é ordens de grandeza maior).
* **O Indicador NSS (Net Sentiment Score):** Funciona como um "saldo bancário" da imagem. A fórmula: `(Positivas − Negativas) / Total × 100`. Se temos 100 notícias, 60 positivas e 40 negativas, o NSS é **+20** (saldo favorável de 20%). Valores acima de zero são saudáveis; abaixo de zero exigem alerta.


In [1]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency
import warnings
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = "vscode"
import plotly.express as px

warnings.filterwarnings('ignore')

# Configuração de visualização
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("Set2")
%matplotlib inline

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

print("✅ Bibliotecas carregadas")

✅ Bibliotecas carregadas


In [2]:
# URL configurada para exportação
url = "https://docs.google.com/spreadsheets/d/1UVGM5g7A2pSmg4Nn5eTzjZhd25sAFFDbkBclRfyNgX8/export?format=csv"

# 1. Carregar dados (leitura inicial como string para evitar inferências erradas)
df = pd.read_csv(url)

# 2. Conversão Robusta para Datas
# format='mixed' resolve o problema de ter '2025-12-31' e '31/3/2024' na mesma coluna
# dayfirst=True ajuda o tradutor a priorizar o dia no formato brasileiro quando houver ambiguidade
df['Data'] = pd.to_datetime(df['Data'], dayfirst=True, format='mixed', errors='coerce')

# 3. Limpeza Geoespacial/Estatística
# Removemos registros onde a data não pôde ser convertida (NaT) para não enviesar o período
df = df.dropna(subset=['Data'])

# 4. Ordenação (Fundamental para séries temporais)
df = df.sort_values('Data')

print(f"Dataset carregado: {df.shape[0]:,} linhas × {df.shape[1]} colunas")
print(f"Período Real Detectado: {df['Data'].min().strftime('%d/%m/%Y')} a {df['Data'].max().strftime('%d/%m/%Y')}")

# Verificação das primeiras e últimas linhas para validar o intervalo
print("\nPrimeiras 3 datas (Início):")
print(df['Data'].head(3))
print("\nÚltimas 3 datas (Fim):")
print(df['Data'].tail(3))


# 4. ENGENHARIA DE PESOS (TIER)
tier_weights = {
    'Muito Relevante': 8,
    'Relevante': 4,
    'Menos Relevante': 1,
    'Null': 1
}
df['Peso'] = df['Tier'].map(tier_weights).fillna(1)

print(f"✅ Dados Carregados e Limpos. Total orgânico: {len(df):,} menções.")

Dataset carregado: 149,514 linhas × 18 colunas
Período Real Detectado: 31/03/2024 a 31/12/2025

Primeiras 3 datas (Início):
78431   2024-03-31
78452   2024-03-31
78453   2024-03-31
Name: Data, dtype: datetime64[ns]

Últimas 3 datas (Fim):
78294   2025-12-31
78284   2025-12-31
78425   2025-12-31
Name: Data, dtype: datetime64[ns]
✅ Dados Carregados e Limpos. Total orgânico: 149,514 menções.


---

> **📌 Filtro aplicado:** A célula abaixo filtra os dados para um grupo específico. Para analisar outro grupo, basta alterar o nome na linha `df = df[df['Grupo'] == '...']`. Para analisar todos os grupos simultaneamente, comente a linha com `#`.


In [3]:
df = df[df['Grupo'] == 'Corsan']


## 1. O Panorama Geral e a Agenda da Mídia

Antes de aprofundarmos no impacto, precisamos entender o cenário macro: como o sentimento está distribuído e quais são os assuntos que pautam a imprensa.

In [10]:
# ======================================================================
# SEÇÃO 1: DISTRIBUIÇÃO DE SENTIMENTOS E AGENDA TEMÁTICA
# ======================================================================

# 1.1 Paleta Global de Cores (Padrão Sentiment Analysis)
COLORS_SENTIMENT = {
    'POSITIVA': '#2ecc71',    # Verde - Sentimento favorável
    'NEUTRA': '#95a5a6',      # Cinza - Sentimento neutro/informativo
    'NEGATIVA': '#e74c3c',    # Vermelho - Sentimento desfavorável
    'PUBLICIDADE': '#3498db'  # Azul - Conteúdo pago (quando aplicável)
}

# 1.2 Função: Distribuição Global de Sentimentos
def plot_classificacao_dinamica(class_counts: pd.Series) -> None:
    """
    Gera visualização dual da distribuição de sentimentos:
    - Gráfico de barras (frequência absoluta)
    - Gráfico de pizza (proporção relativa)
    
    Parâmetros:
        class_counts (pd.Series): Contagem de classificações (POSITIVA/NEUTRA/NEGATIVA)
    """
    # Mapeamento de cores com fallback para categorias não previstas
    color_list = [
        COLORS_SENTIMENT.get(str(label).upper(), '#bdc3c7') 
        for label in class_counts.index
    ]
    
    # Estrutura de subplots: barras + pizza
    fig = make_subplots(
        rows=1, cols=2, 
        specs=[[{"type": "xy"}, {"type": "domain"}]],
        subplot_titles=("Frequência Absoluta", "Proporção (Share de Sentimento)")
    )
    
    # Subplot 1: Barras (volumes)
    fig.add_trace(
        go.Bar(
            x=class_counts.index, 
            y=class_counts.values, 
            marker_color=color_list, 
            showlegend=False
        ), 
        row=1, col=1
    )
    
    # Subplot 2: Pizza (proporções)
    fig.add_trace(
        go.Pie(
            labels=class_counts.index, 
            values=class_counts.values, 
            marker=dict(colors=color_list), 
            hole=0.3,  # Donut chart
            textinfo='percent+label'
        ), 
        row=1, col=2
    )
    
    fig.update_layout(
        title_text="<b>Distribuição Global de Sentimentos</b><br><sup>Visão Geral: Volume vs Proporção</sup>",
        template="plotly_white", 
        height=500
    )
    
    fig.show()


# 1.3 Função: Top Categorias Temáticas
def plot_top_categorias(df_input: pd.DataFrame, n: int = 20) -> None:
    """
    Exibe as N categorias temáticas mais comentadas na mídia.
    Orientação horizontal para facilitar leitura de rótulos longos.
    
    Parâmetros:
        df_input (pd.DataFrame): DataFrame com coluna 'Categoria'
        n (int): Quantidade de categorias a exibir (default=10)
    """
    # Agregação e preparação
    top_cat = df_input['Categoria'].value_counts().head(n).reset_index()
    top_cat.columns = ['Categoria', 'Contagem']
    top_cat = top_cat.sort_values(by='Contagem', ascending=True)  # Menor→Maior (visual melhor)
    
    # Gráfico de barras horizontal
    fig = px.bar(
        top_cat, 
        y='Categoria', 
        x='Contagem', 
        orientation='h', 
        text='Contagem',
        title=f'<b>Agenda Temática: Top {n} Categorias Mais Comentadas</b><br><sup>O que a mídia está falando?</sup>',
        color='Contagem', 
        color_continuous_scale='Viridis'
    )
    
    fig.update_layout(
        template="plotly_white", 
        height=500, 
        margin=dict(l=150)  # Margem esquerda para rótulos longos
    )
    
    fig.show()


# ======================================================================
# EXECUÇÃO: Gerar Visualizações
# ======================================================================

# Gráfico 1: Distribuição de sentimentos
plot_classificacao_dinamica(df['Classificação'].value_counts())

# Gráfico 2: Agenda temática
plot_top_categorias(df)

## 2. A Morfologia da Imprensa: Concentração vs. Pulverização

Nem todos os veículos falam com a mesma frequência. Para entender como a informação se espalha, usamos o conceito de **Cauda Longa**.

**Como ler o gráfico abaixo:**
* **Eixo Horizontal (Ranking):** Lista os veículos do que mais publica (1º) ao que menos publica (últimos).
* **Eixo Vertical (Volume):** Quantidade de matérias.
* **O que analisar:** Observe a linha pontilhada do "Top 20". Um grupo muito pequeno faz muito barulho (cabeça), enquanto centenas de pequenos sites publicam pouco, mas juntos formam um grande volume de pulverização (cauda). Observe qual cor domina a cabeça e a cauda.

In [ ]:
def plot_visualizacao_cauda_longa(df_input):
    df_w = df_input[~df_input['Veículo_de_comunicacao'].isin(['Geral', 'N/A', 'Vários'])]
    agrupado = df_w.groupby('Veículo_de_comunicacao').agg(
        Total=('Classificação', 'count'),
        Positiva=('Classificação', lambda x: (x == 'POSITIVA').sum()),
        Negativa=('Classificação', lambda x: (x == 'NEGATIVA').sum())
    ).sort_values('Total', ascending=False).reset_index()
    agrupado['Ranking'] = agrupado.index + 1
    
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=agrupado['Ranking'], y=agrupado['Positiva'], fill='tozeroy', line=dict(color='#2ecc71'), name='Positivas', text=agrupado['Veículo_de_comunicacao'], hovertemplate='<b>%{text}</b><br>Positivas: %{y}<extra></extra>'))
    fig.add_trace(go.Scatter(x=agrupado['Ranking'], y=agrupado['Negativa'], fill='tozeroy', line=dict(color='#e74c3c'), name='Negativas', text=agrupado['Veículo_de_comunicacao'], hovertemplate='<b>%{text}</b><br>Negativas: %{y}<extra></extra>'))
    
    fig.add_vline(x=20, line_dash="dash", line_color="#2c3e50", annotation_text="← Top 20 | Cauda Longa →", annotation_position="top right")
    fig.update_layout(title='Curva de Cauda Longa da Imprensa', template='plotly_white', hovermode='x unified', height=500, xaxis_title='Ranking do Veículo')
    fig.show()

plot_visualizacao_cauda_longa(df)

## 3. O Efeito "Megafone": A Importância da Relevância (Tier)

Uma crise em veículos "Menos Relevantes" é contornável, mas em "Muito Relevantes" afeta a sociedade em escala macro.

**O Gráfico de NSS Simples vs Ponderado — como ler:**
* A **linha Verde (Simples)** trata todos os veículos como iguais — cada menção vale 1, seja da TV Globo ou de um blog de bairro.
* A **linha Vermelha Tracejada (Ponderada)** multiplica cada menção pelo peso do veículo (Muito Relevante = 8×, Relevante = 4×, Menos Relevante = 1×).
* **Se a linha vermelha está abaixo da verde:** os grandes veículos (megafones) estão batendo na marca, mesmo que a maioria dos sites pequenos esteja elogiando. Quanto maior a distância entre as linhas, maior o "efeito megafone".
* **Se as linhas se aproximam ou cruzam em algum mês:** algo mudou naquele período — possivelmente uma crise que atingiu veículos grandes, ou uma ação de comunicação que alcançou a mídia de massa.
* **A linha cinza pontilhada no zero** é a fronteira: acima = saldo positivo, abaixo = crise.


In [ ]:
# ======================================================================
# CÉLULA DE CÁLCULO - NSS SIMPLES VS PONDERADO (DEFENSIVA)
# ======================================================================
import pandas as pd
import plotly.graph_objects as go

def calcular_comparativo_nss(df_input):
    """
    Calcula a divergência entre o volume bruto e o impacto real (Tier).
    Implementa programação defensiva para garantir a execução.
    """
    # Protege o DataFrame original
    df_temp = df_input.copy()
    
    # 1. VERIFICAÇÃO E CORREÇÃO TEMPORAL
    if 'Ano_Mes' not in df_temp.columns:
        print("🔧 Aviso: Coluna 'Ano_Mes' ausente. Recriando pipeline temporal...")
        # Garante que Data é datetime
        df_temp['Data'] = pd.to_datetime(df_temp['Data'], errors='coerce')
        # Cria a coluna Ano_Mes
        df_temp['Ano_Mes'] = df_temp['Data'].dt.to_period('M').astype(str)
        
    # 2. VERIFICAÇÃO E CORREÇÃO DE PESOS (TIER)
    if 'Peso' not in df_temp.columns:
        print("🔧 Aviso: Coluna 'Peso' ausente. Recriando pipeline de pesos...")
        tier_w = {'Muito Relevante': 8, 'Relevante': 4, 'Menos Relevante': 1, 'Null': 1}
        if 'Tier' in df_temp.columns:
            df_temp['Peso'] = df_temp['Tier'].map(tier_w).fillna(1)
        else:
            df_temp['Peso'] = 1
            
    res = []
    
    # 3. MOTOR DE CÁLCULO
    # Agora é 100% seguro que 'Ano_Mes' existe
    for mes, group in df_temp.groupby('Ano_Mes'):
        
        # NSS Simples
        pos = (group['Classificação'] == 'POSITIVA').sum()
        neg = (group['Classificação'] == 'NEGATIVA').sum()
        tot = len(group)
        nss_simples = ((pos - neg) / tot * 100) if tot > 0 else 0
        
        # NSS Ponderado
        pos_p = (group[group['Classificação'] == 'POSITIVA']['Peso']).sum()
        neg_p = (group[group['Classificação'] == 'NEGATIVA']['Peso']).sum()
        tot_p = group['Peso'].sum()
        nss_pond = ((pos_p - neg_p) / tot_p * 100) if tot_p > 0 else 0
        
        res.append({'Data': mes, 'NSS_Simples': nss_simples, 'NSS_Ponderado': nss_pond})
        
    return pd.DataFrame(res).sort_values('Data')

# ======================================================================
# EXECUÇÃO E PLOTAGEM
# ======================================================================

# Chama a função recém atualizada
nss_df = calcular_comparativo_nss(df)

# Cria a Figura
fig_nss = go.Figure()

# Traço do Volume Simples
fig_nss.add_trace(go.Scatter(
    x=nss_df['Data'], y=nss_df['NSS_Simples'], 
    mode='lines+markers', name='NSS Simples (Volume Bruto)', 
    line=dict(color='#2ecc71', width=3)
))

# Traço do Impacto Ponderado
fig_nss.add_trace(go.Scatter(
    x=nss_df['Data'], y=nss_df['NSS_Ponderado'], 
    mode='lines+markers', name='NSS Ponderado (Impacto por Tier)', 
    line=dict(color='#e74c3c', width=3, dash='dash'), 
    marker=dict(symbol='diamond', size=8)
))

fig_nss.add_hline(y=0, line_dash="dot", line_color="gray", annotation_text="Neutro (NSS=0)")

fig_nss.update_layout(
    title='<b>Evolução do NSS: O Saldo de Imagem (Simples vs Ponderado)</b>', 
    template="plotly_white", 
    hovermode="x unified", 
    height=500
)

# Renderiza o gráfico
fig_nss.show()

🔧 Aviso: Coluna 'Ano_Mes' ausente. Recriando pipeline temporal...


## 4. Radiografia dos Atores: Matriz Espacial

Para agir cirurgicamente, precisamos localizar os veículos num mapa estratégico de risco.

**Como ler a Matriz Espacial de Dispersão:**
* **Tamanho da Bolha:** Quantidade total de matérias publicadas.
* **Posição Cima/Baixo:** Acima da linha tracejada do zero (Aliados). Abaixo da linha (Críticos).
* **Posição Esquerda/Direita:** Quanto mais para a direita, maior o barulho.
* **Onde focar:** O quadrante **inferior direito** é a Zona de Perigo (Alto Volume e Sentimento Negativo).

In [ ]:
def plot_matriz_espacial(df_input, min_n=1):
    prof = df_input.groupby("Veículo_de_comunicacao").agg(
        Total=("Classificação", "count"),
        Positiva=("Classificação", lambda x: (x == "POSITIVA").sum()),
        Negativa=("Classificação", lambda x: (x == "NEGATIVA").sum())
    )
    prof = prof[prof["Total"] >= min_n].copy()
    prof["NSS"] = ((prof["Positiva"] - prof["Negativa"]) / prof["Total"] * 100).round(1)
    prof = prof.reset_index()
    
    if len(prof) == 0: return

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=prof["Total"], y=prof["NSS"], mode="markers",
        marker=dict(size=prof["Total"], sizemode='area', sizeref=2.*max(prof["Total"])/(50.**2), sizemin=8, color=prof["NSS"], colorscale=[[0, "#e74c3c"], [0.5, "#bdc3c7"], [1, "#2ecc71"]], showscale=True),
        text=prof["Veículo_de_comunicacao"], hovertemplate="<b>%{text}</b><br>Volume: %{x}<br>NSS: %{y}%<extra></extra>"
    ))
    fig.add_hline(y=0, line_dash="dash", line_color="#34495e")
    fig.update_layout(title=f"Matriz Espacial: Sentimento × Volume (Mínimo de {min_n} menções)", xaxis_title="Volume Absoluto (Barulho)", yaxis_title="NSS (Sentimento)", template="plotly_white", height=600)
    fig.show()

plot_matriz_espacial(df)

## 5. Dinâmica Temporal e Z-Score (Detector de Crises)

O gráfico diário ou mensal comum oscila muito. Para identificar crises estatisticamente válidas, usamos o **Z-Score** (Desvio Padrão).

**O que é o Z-Score?** Mede quantos desvios-padrão um valor está da média histórica. Na prática: se o volume de negativas em um mês está a +2 desvios da média, significa que esse volume é tão alto que só aconteceria por acaso em ~5% dos meses. É um evento fora do normal.

**Como ler:** A área entre as linhas pontilhadas (+2 e −2) é a "normalidade histórica" — 95% dos meses devem ficar dentro dessa faixa. Se a linha vermelha (negativas) ou verde (positivas) **rompe o limite de ±2**, ocorreu uma **anomalia estatística** — um evento que não pode ser explicado pelo acaso e que merece investigação contextual: o que aconteceu naquele mês?


In [ ]:
# ======================================================================
# CÉLULA DE CÁLCULO - Z-SCORE TEMPORAL (DEFENSIVA)
# ======================================================================
import pandas as pd
import plotly.graph_objects as go

def calcular_zscore_temporal(df_input):
    """
    Calcula anomalias estatísticas (Z-Score) do sentimento midiático.
    Implementa programação defensiva para mitigar erros de estado do Jupyter.
    """
    # 1. Proteção do DataFrame original
    df_temp = df_input.copy()
    
    # 2. VERIFICAÇÃO E CORREÇÃO TEMPORAL
    if 'Ano_Mes' not in df_temp.columns:
        print("🔧 Aviso: Coluna 'Ano_Mes' ausente. Recriando pipeline temporal para o Z-Score...")
        # Se 'Data' existir, recriamos a topologia temporal
        if 'Data' in df_temp.columns:
            df_temp['Data'] = pd.to_datetime(df_temp['Data'], errors='coerce')
            df_temp['Ano_Mes'] = df_temp['Data'].dt.to_period('M').astype(str)
        else:
            raise KeyError("Erro crítico: A coluna base 'Data' também não foi encontrada no dataset.")

    # 3. MOTOR DE CÁLCULO ESTATÍSTICO
    # Agrupa contagem por mês e sentimento
    temp = df_temp.groupby(['Ano_Mes', 'Classificação']).size().unstack(fill_value=0).reset_index()
    
    # Garante que todas as colunas essenciais existam (evita KeyError em meses muito específicos)
    for col in ['POSITIVA', 'NEGATIVA', 'NEUTRA']:
        if col not in temp.columns: 
            temp[col] = 0
            
    # Calcula o Z-Score (Desvio Padrão)
    for col in ['POSITIVA', 'NEGATIVA', 'NEUTRA']:
        mean = temp[col].mean()
        std = temp[col].std() if temp[col].std() > 0 else 1 # Evita divisão por zero
        temp[f'Z_{col}'] = (temp[col] - mean) / std
        
    return temp

# ======================================================================
# EXECUÇÃO E PLOTAGEM
# ======================================================================

# Chamada protegida
df_zscore = calcular_zscore_temporal(df)

fig_zscore = go.Figure()

# Configuração das linhas
series = [
    ('Z_POSITIVA', 'Anomalias Positivas (+)', '#2ecc71'), 
    ('Z_NEGATIVA', 'Anomalias Negativas (-)', '#e74c3c')
]

for col, name, color in series:
    if col in df_zscore.columns:
        fig_zscore.add_trace(go.Scatter(
            x=df_zscore['Ano_Mes'], 
            y=df_zscore[col], 
            mode='lines+markers', 
            name=name, 
            line=dict(color=color, width=3),
            marker=dict(size=8)
        ))

# Linhas de Referência Estatística
fig_zscore.add_hline(y=2, line_dash="dot", line_color="#c0392b", annotation_text="Alerta: Anomalia de Alto Volume (+2σ)")
fig_zscore.add_hline(y=-2, line_dash="dot", line_color="#2980b9", annotation_text="Anomalia de Baixo Volume (-2σ)")
fig_zscore.add_hline(y=0, line_dash="solid", line_color="black", opacity=0.3, annotation_text="Média Histórica (0)")

fig_zscore.update_layout(
    title="<b>Detector de Crises: Z-Score (Desvios Padrão do Sentimento)</b>", 
    template="plotly_white", 
    hovermode="x unified", 
    height=550,
    yaxis_title="Z-Score (Desvios Padrão: σ)",
    xaxis_title="Período"
)

fig_zscore.show()

🔧 Aviso: Coluna 'Ano_Mes' ausente. Recriando pipeline temporal para o Z-Score...


## 6. Mapeamento de Atores: Quem Promove e Quem Detrata

Para atuar no território midiático, precisamos dar "nomes aos bois". Quais veículos são os maiores promotores da marca e quais são os maiores detratores?

**Como ler os gráficos abaixo:**
* São dois gráficos de barras divergentes, cada um com os **Top 20 veículos** ranqueados por volume absoluto de publicações.
* A **barra vermelha (esquerda)** mostra quantas menções negativas aquele veículo publicou. A **barra verde (direita)** mostra as positivas.
* A diferença visual entre os lados revela a polaridade: se a barra vermelha é muito maior que a verde, o veículo é um detrator consistente.
* O ícone ao lado do nome indica a relevância (🔴 Muito Relevante = alto impacto, 🟠 Relevante, 🔵 Menos Relevante, ⚪ Não classificado).
* **Onde focar:** Veículos com 🔴 e barra vermelha grande são prioridade máxima — alto impacto E sentimento negativo.


In [ ]:
def plot_piramides_veiculo_absoluto(df_input: pd.DataFrame, top_n: int = 20) -> None:
    """
    Gera dois gráficos separados (Top N Negativos e Top N Positivos) em Volume Absoluto.
    Implementa tipagem estrita e validação defensiva contra perda de estado do Kernel.
    """
    # 1. Proteção da memória
    df_work = df_input.copy()
    
    # 2. Validação Defensiva de Colunas
    colunas_req = ['Veículo_de_comunicacao', 'Classificação', 'Tier']
    for col in colunas_req:
        if col not in df_work.columns:
            print(f"⚠️ Erro: Coluna '{col}' ausente. Verifique o pipeline de dados.")
            return
            
    # Tratamento de nulos para evitar quebra no agrupamento
    df_work['Tier'] = df_work['Tier'].fillna('Null')
    df_work['Veículo_de_comunicacao'] = df_work['Veículo_de_comunicacao'].fillna('Desconhecido')
    
    # Filtro de veículos genéricos
    veiculos_excluidos = ['Geral', 'N/A', 'Vários']
    df_work = df_work[~df_work['Veículo_de_comunicacao'].isin(veiculos_excluidos)]
    
    # 3. Agrupamento Absoluto
    profile = df_work.groupby('Veículo_de_comunicacao').agg(
        Total=('Classificação', 'count'),
        Positiva=('Classificação', lambda x: (x == 'POSITIVA').sum()),
        Negativa=('Classificação', lambda x: (x == 'NEGATIVA').sum()),
        Tier_Dominante=('Tier', lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else 'Null')
    ).reset_index()

    tier_markers = {'Muito Relevante': '🔴', 'Relevante': '🟠', 'Menos Relevante': '🔵', 'Null': '⚪'}

    # 4. Motor de Renderização
    def criar_piramide(df_plot, title):
        # Mapeia os ícones de Tier de forma segura com .get()
        labels = df_plot.apply(
            lambda r: f"{tier_markers.get(r.get('Tier_Dominante', 'Null'), '⚪')} {r['Veículo_de_comunicacao']}", axis=1
        ).tolist()
        
        fig = go.Figure()
        
        # TRACE NEGATIVO
        fig.add_trace(go.Bar(
            y=labels, x=-df_plot['Negativa'].values, orientation='h',
            name='Menções Negativas', marker_color='#e74c3c',
            text=df_plot['Negativa'].apply(lambda v: f'{v:.0f}' if v > 0 else ''),
            textposition='inside', textfont=dict(color='white', size=11, family='Arial Black'),
            hovertemplate='<b>%{y}</b><br>Negativas: %{customdata[0]:.0f}<extra></extra>',
            customdata=df_plot[['Negativa']].values,
        ))
        
        # TRACE POSITIVO
        fig.add_trace(go.Bar(
            y=labels, x=df_plot['Positiva'].values, orientation='h',
            name='Menções Positivas', marker_color='#2ecc71',
            text=df_plot['Positiva'].apply(lambda v: f'{v:.0f}' if v > 0 else ''),
            textposition='inside', textfont=dict(color='white', size=11, family='Arial Black'),
            hovertemplate='<b>%{y}</b><br>Positivas: %{customdata[0]:.0f}<extra></extra>',
            customdata=df_plot[['Positiva']].values,
        ))
        
        fig.add_vline(x=0, line_width=2, line_color='#2c3e50')
        
        # Engenharia Matemática do Eixo: remove os sinais de negativo (-)
        max_val = max(df_plot['Positiva'].max(), df_plot['Negativa'].max())
        if pd.isna(max_val) or max_val <= 0: max_val = 10
        step = int(np.ceil(max_val / 4 / 10) * 10) 
        step = max(step, 5) 
        
        tickvals = list(range(-step * 4, step * 5, step))
        ticktext = [str(abs(v)) for v in tickvals]

        # Ajuste espacial do canvas
        altura_dinamica = max(500, len(df_plot) * 35)

        fig.update_layout(
            title=title, template='plotly_white', height=altura_dinamica, barmode='relative',
            xaxis=dict(title='Volume Absoluto de Menções', tickvals=tickvals, ticktext=ticktext, zeroline=False),
            yaxis=dict(title='', tickfont=dict(size=11)), margin=dict(l=320, r=30, t=60, b=40),
            legend=dict(orientation='h', y=-0.08, x=0.3)
        )
        return fig

    # Renderiza TOP Negativos
    df_ancora_neg = profile.nlargest(top_n, 'Negativa').sort_values('Negativa', ascending=True)
    if not df_ancora_neg.empty:
        fig_neg = criar_piramide(df_ancora_neg, f'<b>Pirâmide de Crise: Top {top_n} Veículos (Maior Volume NEGATIVO)</b>')
        fig_neg.show()

    # Renderiza TOP Positivos
    df_ancora_pos = profile.nlargest(top_n, 'Positiva').sort_values('Positiva', ascending=True)
    if not df_ancora_pos.empty:
        fig_pos = criar_piramide(df_ancora_pos, f'<b>Pirâmide de Sucesso: Top {top_n} Veículos (Maior Volume POSITIVO)</b>')
        fig_pos.show()

# Chamada da função (usando df_sentiment ou df, garantimos que a função se adapte)
plot_piramides_veiculo_absoluto(df, top_n=20)

## 7. Escala Espacial: Como os Temas variam por Relevância (Treemap)

Na geografia estrutural, sabemos que um fenômeno observado em escala macro comporta-se de maneira diferente na escala micro. O **Treemap** abaixo faz exatamente essa análise espacial das pautas.

**Como ler o Mapa de Árvore:**
* A hierarquia primária são os grandes blocos cinzas: o **Tier** do veículo.
* Dentro de cada bloco, temos as caixas que representam as **Categorias** (Assuntos).
* **Tamanho da caixa:** Mostra o volume de matérias.
* **Cor da caixa:** Mostra o NSS (Sentimento). Vermelho indica crise; Verde indica saldo positivo.
* **A grande sacada:** Compare a cor de uma mesma Categoria (ex: "Abastecimento") dentro do bloco "Muito Relevante" e depois no bloco "Menos Relevante". Isso revela se o atrito está focado na mídia de massa ou na mídia pulverizada local.

In [ ]:
import plotly.express as px

def plot_treemap_tier_categoria(df_input: pd.DataFrame, min_mentions: int = 10) -> None:
    """
    Treemap com hierarquia Tier > Categoria.
    Avalia a divergência semântica de um mesmo tema em diferentes escalas de mídia.
    """
    df_work = df_input.copy()
    
    # 1. Validação Defensiva (Evita KeyError se o Kernel for limpo)
    colunas_req = ['Tier', 'Categoria', 'Classificação']
    for col in colunas_req:
        if col not in df_work.columns:
            print(f"⚠️ Erro: Coluna '{col}' ausente. O gráfico não pode ser gerado.")
            return
            
    # Tratamento seguro de NaNs para agrupamento hierárquico
    df_work['Tier'] = df_work['Tier'].fillna('Null')
    df_work['Categoria'] = df_work['Categoria'].fillna('Sem Categoria classificada')
    
    # 2. Agregação e Cálculo de Sentimento
    treemap_data = df_work.groupby(['Tier', 'Categoria']).agg(
        Total=('Classificação', 'count'),
        Positiva=('Classificação', lambda x: (x == 'POSITIVA').sum()),
        Negativa=('Classificação', lambda x: (x == 'NEGATIVA').sum()),
    ).reset_index()
    
    # Aplica filtro de relevância estatística (ruído visual)
    treemap_data = treemap_data[treemap_data['Total'] >= min_mentions]
    
    if treemap_data.empty:
        print(f"⚠️ Aviso: Nenhuma categoria atingiu o volume mínimo de {min_mentions} menções.")
        return
        
    treemap_data['NSS'] = ((treemap_data['Positiva'] - treemap_data['Negativa']) / treemap_data['Total'] * 100).round(1)
    
    # 3. Ordenação Estrutural (Lógica e não alfabética)
    tier_order = {'Muito Relevante': 1, 'Relevante': 2, 'Menos Relevante': 3, 'Null': 4}
    treemap_data['Tier_Order'] = treemap_data['Tier'].map(tier_order).fillna(5)
    treemap_data = treemap_data.sort_values(['Tier_Order', 'Total'], ascending=[True, False])
    
    # 4. Renderização Espacial
    fig = px.treemap(
        treemap_data,
        path=['Tier', 'Categoria'],
        values='Total',
        color='NSS',
        color_continuous_scale=[
            [0.0, '#c0392b'],   # negativo forte
            [0.3, '#e74c3c'],   # negativo moderado
            [0.5, '#fdfefe'],   # neutro (zero)
            [0.7, '#82e0aa'],   # positivo moderado
            [1.0, '#1e8449'],   # positivo forte
        ],
        color_continuous_midpoint=0,
        title='<b>Mapa Espacial de Sentimento: Escala do Veículo (Tier) × Categoria Temática</b><br>'
              '<sup>Tamanho = Volume | Cor = NSS | Compare a mesma Categoria entre Tiers</sup>',
    )
    
    fig.update_layout(
        template='plotly_white',
        height=700,
        margin=dict(l=10, r=10, t=80, b=10),
        font=dict(family='Inter, Arial', size=12),
        coloraxis_colorbar=dict(title='NSS', ticksuffix=''),
    )
    
    fig.update_traces(
        hovertemplate=(
            '<b>%{label}</b><br>'
            'Total Absoluto: %{value:,} menções<br>'
            'Saldo NSS: %{color:+.1f}<br>'
            '<extra></extra>'
        ),
        textinfo='label+value'
    )
    
    fig.show()

# Chamada da função
plot_treemap_tier_categoria(df, min_mentions=10)

## 8. Evolução Temática: Categoria × Mês (Heatmap Temporal)

Os gráficos anteriores são fotografias — mostram o cenário agregado. Mas os temas problemáticos estão **piorando ou melhorando** ao longo do tempo?

**Como ler o heatmap abaixo:**
* Cada **linha** é uma Categoria temática (Abastecimento, Esgoto, etc.).
* Cada **coluna** é um mês.
* A **cor da célula** mostra o NSS daquele tema naquele mês: vermelho = sentimento negativo, verde = positivo, branco = neutro.
* O **número** dentro de cada célula é o NSS calculado.
* **O que procurar:** Sequências horizontais de vermelho indicam temas com crise persistente. Uma célula vermelha isolada pode ser um evento pontual. Transições de vermelho para verde indicam recuperação de imagem naquele tema.

---


In [17]:
def plot_heatmap_temporal_categoria(df_input: pd.DataFrame, 
                                     top_n: int = 10, 
                                     min_monthly: int = 3) -> None:
    """
    Heatmap temporal: NSS por Categoria × Mês.
    
    Parâmetros:
    - top_n: número de categorias a mostrar (as mais volumosas)
    - min_monthly: mínimo de menções num mês para calcular NSS 
                   (abaixo disso, a célula fica cinza — amostra insuficiente)
    """
    df_work = df_input[df_input['Classificação'] != 'PUBLICIDADE'].copy()
    
    # Garantir coluna temporal
    if 'Ano_Mes' not in df_work.columns:
        df_work['Data'] = pd.to_datetime(df_work['Data'], errors='coerce')
        df_work['Ano_Mes'] = df_work['Data'].dt.to_period('M').astype(str)
    
    # Top categorias por volume
    top_cats = df_work['Categoria'].value_counts().head(top_n).index.tolist()
    df_work = df_work[df_work['Categoria'].isin(top_cats)]
    
    # Calcular NSS + Positivos + Negativos por Categoria × Mês
    results = []
    for cat in top_cats:
        for mes in sorted(df_work['Ano_Mes'].unique()):
            subset = df_work[(df_work['Categoria'] == cat) & (df_work['Ano_Mes'] == mes)]
            n = len(subset)
            pos = (subset['Classificação'] == 'POSITIVA').sum()
            neg = (subset['Classificação'] == 'NEGATIVA').sum()
            
            if n >= min_monthly:
                nss = round((pos - neg) / n * 100, 1)
            else:
                nss = np.nan  # Amostra insuficiente
            
            results.append({
                'Categoria': cat, 
                'Mês': mes, 
                'NSS': nss, 
                'N': n,
                'Positivos': pos,
                'Negativos': neg
            })
    
    heat_df = pd.DataFrame(results)
    heat_pivot = heat_df.pivot(index='Categoria', columns='Mês', values='NSS')
    n_pivot = heat_df.pivot(index='Categoria', columns='Mês', values='N')
    pos_pivot = heat_df.pivot(index='Categoria', columns='Mês', values='Positivos')
    neg_pivot = heat_df.pivot(index='Categoria', columns='Mês', values='Negativos')
    
    # Reordenar categorias pelo volume total (mais volumosas no topo)
    heat_pivot = heat_pivot.reindex(top_cats)
    n_pivot = n_pivot.reindex(top_cats)
    pos_pivot = pos_pivot.reindex(top_cats)
    neg_pivot = neg_pivot.reindex(top_cats)
    
    # Preparar customdata: [N, Positivos, Negativos] para cada célula
    customdata = np.dstack((
        n_pivot.values,
        pos_pivot.values,
        neg_pivot.values
    ))
    
    # Anotações com NSS + N
    annotations = []
    for i, cat in enumerate(heat_pivot.index):
        for j, mes in enumerate(heat_pivot.columns):
            nss_val = heat_pivot.iloc[i, j]
            n_val = n_pivot.iloc[i, j]
            if pd.notna(nss_val):
                text = f"{nss_val:+.0f}"
                color = 'white' if abs(nss_val) > 50 else '#1a202c'
            else:
                text = "—"
                color = '#bdc3c7'
            annotations.append(dict(
                x=mes, y=cat, text=text, showarrow=False,
                font=dict(size=10, color=color)
            ))
    
    fig = go.Figure(go.Heatmap(
        z=heat_pivot.values,
        x=heat_pivot.columns.tolist(),
        y=heat_pivot.index.tolist(),
        customdata=customdata,
        colorscale=[
            [0.0, '#c0392b'], [0.3, '#e74c3c'], [0.5, '#fdfefe'],
            [0.7, '#82e0aa'], [1.0, '#1e8449'],
        ],
        zmid=0,
        colorbar=dict(title='NSS'),
        hovertemplate=(
            '<b>%{y}</b><br>'
            'Mês: %{x}<br>'
            'NSS: %{z:+.1f}<br>'
            '<b>Positivos:</b> %{customdata[1]:.0f}<br>'
            '<b>Negativos:</b> %{customdata[2]:.0f}<br>'
            '<b>Total:</b> %{customdata[0]:.0f}<br>'  # ← Total
            '<i>NSS = (Pos - Neg) / Total × 100</i>'  # ← Fórmula
            '<extra></extra>'
        ),
    ))
    
    fig.update_layout(
        title='<b>Evolução Temática: NSS por Categoria × Mês</b><br>'
              '<sup>Vermelho = crise | Verde = saudável | — = amostra insuficiente</sup>',
        template='plotly_white',
        height=max(450, top_n * 40),
        yaxis=dict(autorange='reversed'),
        margin=dict(l=180, r=30, t=80, b=50),
        annotations=annotations,
        font=dict(family='Inter, Arial', size=12),
    )
    
    fig.show()


plot_heatmap_temporal_categoria(df, top_n=10, min_monthly=3)

## 9. Síntese Executiva e Ações Recomendadas

### O que os dados revelam

Este relatório analisou a cobertura de mídia espontânea através de 7 perspectivas complementares. Os principais achados:

1. **O sentimento geral é positivo, mas frágil.** O NSS Simples tende a ser favorável porque a maioria das menções vem de veículos pequenos que reproduzem conteúdo institucional. Quando ponderamos pela relevância do veículo, o cenário muda — o "efeito megafone" mostra que os grandes veículos são mais críticos.

2. **A negatividade é concentrada e temática.** Não é "a mídia em geral" que é negativa — são poucos temas operacionais (abastecimento, esgoto) em poucos veículos de grande porte. Isso é uma boa notícia estratégica: o problema é localizado e, portanto, tratável.

3. **A estrutura de Cauda Longa é uma vantagem.** A pulverização da cobertura em centenas de veículos pequenos funciona como um "colchão" de positividade. Manter o relacionamento com esses veículos protege o NSS Simples, mesmo em períodos de crise na mídia grande.

### Ações sugeridas

| Prioridade | Ação | Baseado em |
|-----------|------|-----------|
| 🔴 Alta | Monitorar e criar protocolo de resposta rápida para os veículos do **quadrante inferior direito** da Matriz Espacial (alto volume + NSS negativo) | Seção 4 |
| 🟠 Média | Investigar os meses onde o Z-Score rompeu ±2 — o que aconteceu operacionalmente nesses períodos? | Seção 5 |
| 🟢 Contínua | Manter o fluxo de releases para os veículos da Cauda Longa — eles sustentam o "colchão" de positividade | Seção 2 |
| 🔵 Estratégica | Comparar o treemap entre Tiers: temas que são verdes no Tier "Menos Relevante" mas vermelhos no "Muito Relevante" indicam oportunidade de reframing da narrativa na mídia grande | Seção 7 |

---


## Apêndice A: Visualização Sunburst (Experimental)

O Sunburst é a versão circular do Treemap. Ambos mostram hierarquias, mas com ênfase diferente:

| Aspecto | Treemap | Sunburst |
|---------|---------|----------|
| Ênfase visual | **Tamanho** (área proporcional ao volume) | **Hierarquia** (drill-down camada por camada) |
| Comparação de volume | Fácil (blocos lado a lado) | Difícil (arcos curvos distorcem percepção de área) |
| Navegação | Clique para entrar/sair do nível | Clique no anel para expandir/colapsar |
| Melhor para | "Onde está o peso?" | "Qual a estrutura interna?" |

**Como ler o Sunburst abaixo:**
* **Anel central:** Classificação (POSITIVA, NEGATIVA, NEUTRA) — o primeiro nível de organização.
* **Anel intermediário:** Categoria — dentro de cada classificação, quais temas aparecem.
* **Anel externo:** Subcategoria — o detalhe final.
* **Tamanho do arco:** Proporção do volume.
* **Interação:** Clique em qualquer fatia para expandir e ver seus detalhes internos. Clique no centro para voltar.

**Quando usar Sunburst em vez de Treemap:** Quando a pergunta é "qual a composição interna de cada classificação?" (do centro para fora). O Treemap responde melhor "como o mesmo tema muda entre Tiers?" (comparação lateral).

---


In [ ]:
def plot_sunburst_composicao(df_input: pd.DataFrame, min_mentions: int = 5) -> None:
    """
    Sunburst: Classificação > Categoria > Subcategoria.
    
    Mostra a composição interna de cada tipo de sentimento.
    Responde: "das menções positivas, quais temas dominam? 
    E dentro de cada tema, quais subtemas?"
    """
    df_work = df_input.copy()
    
    # Excluir Publicidade (não é sentimento orgânico)
    df_work = df_work[df_work['Classificação'] != 'PUBLICIDADE']
    
    # Agregar
    sun_data = df_work.groupby(['Classificação', 'Categoria', 'Subcategoria']).size().reset_index(name='Total')
    sun_data = sun_data[sun_data['Total'] >= min_mentions]
    
    if sun_data.empty:
        print(f"⚠️ Nenhuma combinação atingiu {min_mentions} menções.")
        return
    
    # Paleta de cores por Classificação
    color_map = {
        'POSITIVA': '#2ecc71',
        'NEUTRA': '#95a5a6',
        'NEGATIVA': '#e74c3c',
    }
    
    fig = px.sunburst(
        sun_data,
        path=['Classificação', 'Categoria', 'Subcategoria'],
        values='Total',
        color='Classificação',
        color_discrete_map=color_map,
        title='<b>Sunburst: Composição Temática por Sentimento</b><br>'
              '<sup>Clique nas fatias para expandir | Centro → Classificação → Categoria → Subcategoria</sup>',
    )
    
    fig.update_layout(
        height=700,
        margin=dict(l=10, r=10, t=80, b=10),
        font=dict(family='Inter, Arial', size=12),
    )
    
    fig.update_traces(
        textinfo='label+percent parent',
        hovertemplate='<b>%{label}</b><br>Menções: %{value:,}<br>% do nível acima: %{percentParent:.1%}<extra></extra>',
        insidetextorientation='radial',
    )
    
    fig.show()


# Sunburst com hierarquia Classificação > Categoria > Subcategoria
plot_sunburst_composicao(df, min_mentions=3)


**Variação alternativa:** Sunburst com hierarquia **Tier > Categoria > Classificação**. Aqui o anel central é a relevância do veículo, e a composição de sentimento aparece no anel externo. Permite ver "dentro dos veículos Muito Relevantes, quais temas e qual o sentimento de cada um?"


In [ ]:
def plot_sunburst_tier(df_input: pd.DataFrame, min_mentions: int = 5) -> None:
    """
    Sunburst alternativo: Tier > Categoria > Classificação.
    Mostra a composição de sentimento dentro de cada Tier e tema.
    """
    df_work = df_input[df_input['Classificação'] != 'PUBLICIDADE'].copy()
    df_work['Tier'] = df_work['Tier'].fillna('Null')
    
    sun_data = df_work.groupby(['Tier', 'Categoria', 'Classificação']).size().reset_index(name='Total')
    sun_data = sun_data[sun_data['Total'] >= min_mentions]
    
    if sun_data.empty:
        print(f"⚠️ Nenhuma combinação atingiu {min_mentions} menções.")
        return
    
    color_map = {
        'POSITIVA': '#2ecc71',
        'NEUTRA': '#95a5a6',
        'NEGATIVA': '#e74c3c',
    }
    
    fig = px.sunburst(
        sun_data,
        path=['Tier', 'Categoria', 'Classificação'],
        values='Total',
        color='Classificação',
        color_discrete_map=color_map,
        title='<b>Sunburst: Tier × Categoria × Sentimento</b><br>'
              '<sup>Clique para expandir | Centro = Tier → Categoria → Classificação</sup>',
    )
    
    fig.update_layout(
        height=700,
        margin=dict(l=10, r=10, t=80, b=10),
        font=dict(family='Inter, Arial', size=12),
    )
    
    fig.update_traces(
        textinfo='label+percent parent',
        hovertemplate='<b>%{label}</b><br>Menções: %{value:,}<br>% do nível acima: %{percentParent:.1%}<extra></extra>',
        insidetextorientation='radial',
    )
    
    fig.show()


plot_sunburst_tier(df, min_mentions=3)


In [ ]:
# Injeção de CSS/JS para o botão "Ver Script Python"
OUTPUT_HTML = "Relatorio_Satisfacao_Executivo.html"
style_and_script = """
<style>
    .code-hidden .jp-InputArea { display: none !important; }
    .code-toggle-btn { background: #f8f9fa; border: 1px solid #ddd; padding: 6px 12px; cursor: pointer; border-radius: 4px; font-weight: bold; color: #2c3e50; }
</style>
<script>
document.addEventListener('DOMContentLoaded', function() {
    document.querySelectorAll('.jp-CodeCell').forEach(cell => {
        cell.classList.add('code-hidden');
        let btn = document.createElement("button");
        btn.className = "code-toggle-btn";
        btn.innerHTML = "📊 Exibir/Ocultar Código Fonte";
        btn.onclick = function() { cell.classList.toggle('code-hidden'); };
        cell.insertBefore(btn, cell.firstChild);
    });
});
</script>
"""
print("Exportação pronta! Lembre-se de rodar este script no final e exportar para HTML pelo menu do Jupyter.")

Exportação pronta! Lembre-se de rodar este script no final e exportar para HTML pelo menu do Jupyter.
